In [5]:
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from scipy.special import expit
from adbench.myutils_new import Utils
from adbench.run_new import RunPipeline
import pandas as pd
from numba import njit, prange
from pynndescent import NNDescent


def count_points_within_radius(X, tree, epsilon):
    neighbors = tree.query_ball_tree(tree, epsilon)
    counts = np.array([len(pts) - 1 for pts in neighbors])
    return counts


def max_min_distances_kdtree(X):
    tree = cKDTree(X)
    dists, _ = tree.query(X, k=X.shape[0])
    all_distances = dists[:, 1:].flatten()
    return np.max(all_distances), np.min(all_distances)


def binary_search_condition(low, high, condition, tol=1e-4, max_iter=50):
    result = None
    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        if condition(mid):
            result = mid
            high = mid
        else:
            low = mid
        if abs(high - low) < tol:
            break
    return result


def condition_formulation(point_in_radius_counts, nbd_sample_count_threshold, satisfiability_proportion):
    satisfied = (point_in_radius_counts > nbd_sample_count_threshold).astype(int)
    return np.sum(satisfied) >= satisfiability_proportion


# NEW: Binary k-NN graph (0–1 adjacency)

def binary_knn_graph(X_batch, n_neighbors):
    tree = cKDTree(X_batch)
    _, indices = tree.query(X_batch, k=n_neighbors + 1)

    n = X_batch.shape[0]
    A = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        for j in indices[i, 1:]:  # skip self
            A[i, j] = 1.0
            A[j, i] = 1.0  # symmetric graph

    return A



def get_empirical_weights(
    X,
    nbd_sample_count_threshold=5,
    max_iters_weight_count=4,
    satisfiability_proportion=0.3,
    n_neighbors=15,
    batch_size=1000
):

    def compute_weights_from_graph(G):
        tree = cKDTree(G)
        max_dist, min_dist = max_min_distances_kdtree(G)

        threshold = min(nbd_sample_count_threshold, len(G) - 1)
        effective_required = int(satisfiability_proportion * len(G))

        eps = binary_search_condition(
            min_dist, max_dist,
            lambda r: condition_formulation(
                count_points_within_radius(G, tree, r),
                threshold,
                effective_required
            )
        )

        if eps is None:
            eps = max_dist

        delta = (eps - 1e-6) / max_iters_weight_count
        counts_all = []

        for _ in range(max_iters_weight_count):
            counts_all.append(count_points_within_radius(G, tree, eps))
            eps -= delta

        return np.mean(counts_all, axis=0)

    if len(X) <= 3 * n_neighbors:
        G = binary_knn_graph(X, n_neighbors)
        return compute_weights_from_graph(G)

    weights_all = []
    batch_size = min(batch_size, max(3 * n_neighbors, 100))

    for i in range(0, len(X), batch_size):
        X_batch = X[i:i + batch_size]
        G = binary_knn_graph(X_batch, n_neighbors)
        weights_all.append(compute_weights_from_graph(G))

    return np.concatenate(weights_all)



@njit(fastmath=True, parallel=True)
def shift_data(X, indices, weights, learning_rate):

    n, k = indices.shape
    d = X.shape[1]

    revised = np.empty_like(X)
    change = np.empty(n)

    for i in prange(n):

        denom = 0.0
        for j in range(k):
            denom += weights[indices[i, j]]
        if denom < 1e-6:
            denom = 1e-6

        for t in range(d):
            acc = 0.0
            for j in range(k):
                acc += weights[indices[i, j]] * X[indices[i, j], t]
            revised[i, t] = acc / denom

        dist = 0.0
        for t in range(d):
            diff = revised[i, t] - X[i, t]
            dist += diff * diff
        dist = np.sqrt(dist)
        change[i] = dist

        if dist > 1e-6:
            scale = learning_rate * dist
            for t in range(d):
                revised[i, t] = X[i, t] + scale * (revised[i, t] - X[i, t]) / dist
        else:
            revised[i] = X[i]

    return revised, change



def get_shift_fast(X, k, nbd_sample_count_threshold,
                   learning_rate, max_iters_shift,
                   shift_threshold, return_weights=False):

    weights = get_empirical_weights(
        X,
        nbd_sample_count_threshold=nbd_sample_count_threshold,
        n_neighbors=k
    )

    shifted = X.copy()
    total_distance = np.zeros(X.shape[0])

    for _ in range(max_iters_shift):

        index = NNDescent(
            shifted,
            n_neighbors=k,
            metric="euclidean",
            random_state=42
        )

        indices, _ = index.neighbor_graph
        indices = indices.astype(np.int64)

        shifted, change = shift_data(
            shifted, indices, weights, learning_rate
        )

        total_distance += change
        if change.mean() < shift_threshold:
            break

    if return_weights:
        return shifted, weights, total_distance
    return shifted, total_distance


def mean_shift_manifold_learning(
    X, k=30, nbd_sample_count_threshold=30,
    learning_rate=0.3, max_iters_shift=10,
    shift_threshold=1e-4, return_weights=False
):

    return get_shift_fast(
        X, k, nbd_sample_count_threshold,
        learning_rate, max_iters_shift,
        shift_threshold, return_weights
    )


class MSML:
    def __init__(self, seed: int, model_name: str = 'MSML', k=100, nbd_sample_count_threshold=70, learning_rate=0.1, max_iters_shift=6, shift_threshold=0.003, anomalyThreshold=0.22, scaler=StandardScaler()):
        self.k = k
        self.nbd_sample_count_threshold = nbd_sample_count_threshold
        self.learning_rate = learning_rate
        self.max_iters_shift = max_iters_shift
        self.shift_threshold = shift_threshold
        self.anomalyThreshold = anomalyThreshold
        self.scaler = scaler
        self.seed = seed
        self.utils = Utils()
        self.model_name = model_name

    def fit(self, X_train, y_train=None):
        return self

    def predict_score(self, X):
        data_shifted, total_distance = mean_shift_manifold_learning(
            X,
            self.k,
            self.nbd_sample_count_threshold,
            self.learning_rate,
            self.max_iters_shift,
            self.shift_threshold
        )

        total_distance = self.scaler.fit_transform(total_distance.reshape(-1, 1))
        total_distance = expit(total_distance)
        return total_distance.squeeze()



if __name__ == "__main__":

    utils = Utils()
    utils.download_datasets()

    pipeline = RunPipeline(
        suffix="ADBench",
        parallel="unsupervise",
        realistic_synthetic_mode="local",
        noise_type=None
    )

    results = pipeline.run(clf=MSML)
    pd.DataFrame(results).to_csv("adbench/result/MSDE_simple_graph_local.csv", index=False)

    # results2 = pipeline.run()
    # pd.DataFrame(results2).to_csv("adbench/result/benchmarks.csv", index=False)


if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 647.94it/s]

CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(618), 'Anomalies Ratio(%)': np.float64(6.18)}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(578), 'Anomalies Ratio(%)': np.float64(5.78)}
subsampling for dataset 11_donors...
current noise type: None
{'Samples': 10000, 'Features': 10, 'Anomalies': np.int64(597), 'Anomalies Ratio(%)': np.float64(5.97)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
current noise type: None
{'Samples': 1941, 'Features': 27, 'Anomalies': np.int32(673), 'Anomalies Ratio(%)': np.float64(34.67)}
subsampling for dataset 13_fraud...
current noise type: None
{'Samples': 10000, 'Features': 29, 'Anomalies': np.int64(15)

0it [00:00, ?it/s]

...

Current experiment parameters: ('9_census', 0.0, np.int64(1)), model: Customized, metrics: {'aucroc': np.float64(0.9999942393548077), 'aucpr': np.float64(0.9999129697698855), 'p_at_n': np.float64(0.9945945945945946), 'adj_p_at_n': np.float64(0.9942393548077385), 'adj_ap': np.float64(0.9999072501988123)}, fitting time: 1.1920928955078125e-06, inference time: 9.709388256072998
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(628), 'Anomalies Ratio(%)': np.float64(6.28)}
Model: Customized, AUC-ROC: 0.9999451439121092, AUC-PR: 0.9992821873480543


1082it [25:12,  4.91s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(2)), model: Customized, metrics: {'aucroc': np.float64(0.9999451439121092), 'aucpr': np.float64(0.9992821873480543), 'p_at_n': np.float64(0.9946808510638298), 'adj_p_at_n': np.float64(0.9943252322871583), 'adj_ap': np.float64(0.9992341970285075)}, fitting time: 1.1920928955078125e-06, inference time: 10.097485542297363
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(652), 'Anomalies Ratio(%)': np.float64(6.52)}
Model: Customized, AUC-ROC: 0.9821610527235145, AUC-PR: 0.8775278531991874


1104it [26:30,  1.44s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(3)), model: Customized, metrics: {'aucroc': np.float64(0.9821610527235145), 'aucpr': np.float64(0.8775278531991874), 'p_at_n': np.float64(0.7295918367346939), 'adj_p_at_n': np.float64(0.7106902675478179), 'adj_ap': np.float64(0.8689670326667482)}, fitting time: 1.430511474609375e-06, inference time: 10.953497171401978
